# Set Up Cell

In [22]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torchvision.models import ResNet18_Weights, MobileNet_V2_Weights, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# =====================================================
# SET SEED
# =====================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# ROBUST DATASET
# =====================================================
class ImageFolderDataset(Dataset):
    def __init__(self, folder, transform):
        self.paths = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.transform(img)
        except:
            # fallback: return a random valid image instead
            new_idx = random.randint(0, len(self.paths) - 1)
            img = Image.open(self.paths[new_idx]).convert("RGB")
            return self.transform(img)

# =====================================================
# MODEL
# =====================================================
def load_model(name):
    if name == "resnet18":
        model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features

    elif name == "mobilenetv2":
        model = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        in_features = model.classifier[1].in_features

    elif name == "efficientnetb0":
        model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features

    else:
        raise ValueError("Unsupported model")

    return model.to(device)


# =====================================================
# FEATURE EXTRACTOR
# =====================================================
def get_feature_extractor(model, name):
    if name == "resnet18":
        return nn.Sequential(*list(model.children())[:-1])
    elif name == "mobilenetv2":
        return model.features
    elif name == "efficientnetb0":
        return model.features

# =====================================================
# EMBEDDINGS
# =====================================================
def extract_embeddings(model, loader, name):
    model.eval()
    extractor = get_feature_extractor(model, name)

    embeddings = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            feats = extractor(batch)
            feats = torch.flatten(feats, 1)
            embeddings.append(feats.cpu().numpy())

    return np.concatenate(embeddings, axis=0)


# =====================================================
# EVALUATION FUNCTION
# =====================================================
def evaluate(model, loader_A, loader_B):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for loader, label in [(loader_A, 0), (loader_B, 1)]:
            for batch in loader:
                batch = batch.to(device)
                outputs = model(batch)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                y_pred.extend(preds)
                y_true.extend([label] * len(preds))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')

    return acc, f1

def train_and_evaluate_models(CONFIG):

    set_seed(42)

    os.makedirs(os.path.dirname(CONFIG["output"]["checkpoint_path"]), exist_ok=True)
    
    transform = transforms.Compose([
        transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        transforms.ToTensor()
    ])

    train_A = ImageFolderDataset(CONFIG["data"]["train_A"], transform)
    train_B = ImageFolderDataset(CONFIG["data"]["train_B"], transform)
    
    val_A = ImageFolderDataset(CONFIG["data"]["val_A"], transform)
    val_B = ImageFolderDataset(CONFIG["data"]["val_B"], transform)
    
    test_A = ImageFolderDataset(CONFIG["data"]["test_A"], transform)
    test_B = ImageFolderDataset(CONFIG["data"]["test_B"], transform)
    
    train_loader_A = DataLoader(train_A, batch_size=CONFIG["batch_size"], shuffle=True)
    train_loader_B = DataLoader(train_B, batch_size=CONFIG["batch_size"], shuffle=True)
    
    val_loader_A = DataLoader(val_A, batch_size=CONFIG["batch_size"], shuffle=False)
    val_loader_B = DataLoader(val_B, batch_size=CONFIG["batch_size"], shuffle=False)
    
    test_loader_A = DataLoader(test_A, batch_size=CONFIG["batch_size"], shuffle=False)
    test_loader_B = DataLoader(test_B, batch_size=CONFIG["batch_size"], shuffle=False)
    
    # =====================================================
    # TRAINING
    # =====================================================
    model = load_model(CONFIG["model_name"])
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\n🚀 Epoch {epoch+1}/{CONFIG['epochs']}")
    
        model.train()
    
        for batch_A, batch_B in zip(train_loader_A, train_loader_B):
            batch_A = batch_A.to(device)
            batch_B = batch_B.to(device)
    
            inputs = torch.cat([batch_A, batch_B], dim=0)
            labels = torch.cat([
                torch.zeros(len(batch_A)),
                torch.ones(len(batch_B))
            ]).long().to(device)
    
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
        # =====================================================
        # VALIDATION CHECKPOINTING
        # =====================================================
        val_acc, val_f1 = evaluate(model, val_loader_A, val_loader_B)
    
        print(f"Val Accuracy: {val_acc:.4f}, Val F1: {val_f1:.4f}")
    
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc
            }, CONFIG["output"]["checkpoint_path"])
    
            print("✅ Saved new best model!")
    
        model_folder, model_filename = os.path.split(CONFIG["output"]["checkpoint_path"])
        model_filename, ext = os.path.splitext(model_folder)
        epoch_wise_model_filename = model_filename + f'_epoch_{epoch}' + '.' + ext
        epoch_wise_model_filepath = os.path.join(model_folder, epoch_wise_model_filename)
    
        # torch.save({
        #         'epoch': epoch,
        #         'model_state_dict': model.state_dict(),
        #         'optimizer_state_dict': optimizer.state_dict(),
        #         'val_acc': val_acc
        # }, epoch_wise_model_filepath)
    
    # =====================================================
    # FINAL EVALUATION
    # =====================================================
    train_acc, train_f1 = evaluate(model, train_loader_A, train_loader_B)
    test_acc, test_f1 = evaluate(model, test_loader_A, test_loader_B)
    
    print("\n📊 FINAL RESULTS")
    
    print("\n--- TRAIN ---")
    print(f"Accuracy: {train_acc:.4f}")
    print(f"F1 Score: {train_f1:.4f}")
    
    print("\n--- TEST ---")
    print(f"Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {test_f1:.4f}")


# Apples And Oranges

## Resnet-18 Model Training

In [2]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_resnet18.pth",
        "plot_path": "/kaggle/working/skin_lesion_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 186MB/s]



🚀 Epoch 1/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 3/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 4/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 5/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 6/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 7/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 8/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 9/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 10/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 11/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 12/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 13/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 14/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 15/15
Val Accuracy: 0.9000, Val F1: 0.8990

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.8603
F1 Score: 0.8571


## MobileNet-v2 Model Training

In [3]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_mobilenetv2.pth",
        "plot_path": "/kaggle/working/skin_lesion_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 117MB/s] 



🚀 Epoch 1/15
Val Accuracy: 0.4500, Val F1: 0.5466
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.8500, Val F1: 0.8465
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.9000, Val F1: 0.8990
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 5/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 6/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 7/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 8/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 9/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 10/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 11/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 12/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 13/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 14/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 15/15
Val Accuracy: 0.9000, Val F1: 0.8990

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.7166
F1 Score: 0.6911


## EfficientNet-B0 Model Training

In [4]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/apples_and_oranges_efficientnetb0.pth",
        "plot_path": "/kaggle/working/apples_and_oranges_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 128MB/s] 



🚀 Epoch 1/15
Val Accuracy: 0.7000, Val F1: 0.7857
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.8500, Val F1: 0.8663
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 4/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 5/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 6/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 7/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 8/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 9/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 10/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 11/15
Val Accuracy: 0.9000, Val F1: 0.8990
✅ Saved new best model!

🚀 Epoch 12/15
Val Accuracy: 0.9000, Val F1: 0.8990

🚀 Epoch 13/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 14/15
Val Accuracy: 0.8500, Val F1: 0.8465

🚀 Epoch 15/15
Val Accuracy: 0.8500, Val F1: 0.8465

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.7915
F1 Score: 0.7856


# Horses And Zebra

## Resnet-18 Model training

In [5]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_resnet18.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.6667, Val F1: 0.6250
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 3/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 4/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 5/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 6/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 7/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 8/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 9/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 10/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 11/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 12/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 13/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 14/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 15/15
Val Accuracy: 0.6333, Val F1: 0.5764

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9750
F1 Score: 0.9751


## Mobiletnet-v2 Model Training

In [6]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_mobilenetv2.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.6000, Val F1: 0.5417
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6333, Val F1: 0.5764
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 4/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 5/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 6/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 7/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 8/15
Val Accuracy: 0.6333, Val F1: 0.5764

🚀 Epoch 9/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 10/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 11/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 12/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 13/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 14/15
Val Accuracy: 0.6000, Val F1: 0.5238

🚀 Epoch 15/15
Val Accuracy: 0.6000, Val F1: 0.5238

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9792
F1 Score: 0.9812


## EfficientNet-B0 Model Training

In [7]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_efficientnetb0.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5333, Val F1: 0.4034
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6000, Val F1: 0.5238
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.5667, Val F1: 0.4665

🚀 Epoch 4/15
Val Accuracy: 0.5667, Val F1: 0.4665

🚀 Epoch 5/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 6/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 7/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 8/15
Val Accuracy: 0.5667, Val F1: 0.4665

🚀 Epoch 9/15
Val Accuracy: 0.5667, Val F1: 0.4665

🚀 Epoch 10/15
Val Accuracy: 0.5667, Val F1: 0.4665

🚀 Epoch 11/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 12/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 13/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 14/15
Val Accuracy: 0.5333, Val F1: 0.4034

🚀 Epoch 15/15
Val Accuracy: 0.5333, Val F1: 0.4034

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9542
F1 Score: 0.9543


# Skin Lesion

In [8]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_resnet18.pth",
        "plot_path": "/kaggle/working/skin_lesion_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.4000, Val F1: 0.4211
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.8000, Val F1: 0.8750
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9767
F1 Score: 0.9767


In [9]:
BASE_DIR  = r'//kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_mobilenetv2.pth",
        "plot_path": "/kaggle/working/skin_lesion_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.0000, Val F1: 0.0000

🚀 Epoch 2/15
Val Accuracy: 0.0000, Val F1: 0.0000

🚀 Epoch 3/15
Val Accuracy: 0.3500, Val F1: 0.4298
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.8000, Val F1: 0.7917
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 0.9500, Val F1: 0.9499
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 7/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 8/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 9/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 10/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 11/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 12/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 13/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 14/15
Val Accuracy: 0.9500, Val F1: 0.9499

🚀 Epoch 15/15
Val Accuracy: 0.9500, Val F1: 0.9499

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9535
F1 Score: 0.9535


In [10]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_efficientnetb0.pth",
        "plot_path": "/kaggle/working/skin_lesion_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.0000, Val F1: 0.0000

🚀 Epoch 2/15
Val Accuracy: 0.1500, Val F1: 0.2576
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.1500, Val F1: 0.1875

🚀 Epoch 4/15
Val Accuracy: 0.4500, Val F1: 0.4720
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 0.6000, Val F1: 0.5769
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 0.6000, Val F1: 0.6000

🚀 Epoch 7/15
Val Accuracy: 0.6500, Val F1: 0.6267
✅ Saved new best model!

🚀 Epoch 8/15
Val Accuracy: 0.6500, Val F1: 0.6267

🚀 Epoch 9/15
Val Accuracy: 0.7000, Val F1: 0.6875
✅ Saved new best model!

🚀 Epoch 10/15
Val Accuracy: 0.7000, Val F1: 0.7038

🚀 Epoch 11/15
Val Accuracy: 0.7500, Val F1: 0.7333
✅ Saved new best model!

🚀 Epoch 12/15
Val Accuracy: 0.7500, Val F1: 0.7333

🚀 Epoch 13/15
Val Accuracy: 0.7500, Val F1: 0.7333

🚀 Epoch 14/15
Val Accuracy: 0.7500, Val F1: 0.7442

🚀 Epoch 15/15
Val Accuracy: 0.7500, Val F1: 0.7333

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accur

# Watermark Dataset

In [11]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_resnet18.pth",
        "plot_path": "/kaggle/working/watermark_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9889
F1 Score: 0.9889


In [12]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_mobilenetv2.pth",
        "plot_path": "/kaggle/working/watermark_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.1000, Val F1: 0.1667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6500, Val F1: 0.7778
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.9000, Val F1: 0.9206
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.9500, Val F1: 0.9499
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9667
F1 Score: 0.

In [13]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_efficientnetb0.pth",
        "plot_path": "/kaggle/working/watermark_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.1000, Val F1: 0.1667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.9000, Val F1: 0.9444
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.9500, Val F1: 0.9737
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9556
F1 Score: 0.9557


# Pneumonia CXR

In [23]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_B),
        "val_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_A),
        "val_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_B),
        "test_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_A),
        "test_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_resnet18.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.3636
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6250, Val F1: 0.6827
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.8125, Val F1: 0.8057
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.7500, Val F1: 0.7544

🚀 Epoch 5/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 6/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 7/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 8/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 9/15
Val Accuracy: 0.7500, Val F1: 0.7544

🚀 Epoch 10/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 11/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 12/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 13/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 14/15
Val Accuracy: 0.8125, Val F1: 0.8057

🚀 Epoch 15/15
Val Accuracy: 0.8125, Val F1: 0.8057

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.6490
F1 Score: 0.6392


In [24]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_B),
        "val_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_A),
        "val_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_B),
        "test_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_A),
        "test_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_resnet18.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.2500, Val F1: 0.2222
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.4375, Val F1: 0.4067
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.3333
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 5/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 6/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 7/15
Val Accuracy: 0.5000, Val F1: 0.4182

🚀 Epoch 8/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 9/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 10/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 11/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 12/15
Val Accuracy: 0.4375, Val F1: 0.3043

🚀 Epoch 13/15
Val Accuracy: 0.5000, Val F1: 0.4182

🚀 Epoch 14/15
Val Accuracy: 0.5000, Val F1: 0.4182

🚀 Epoch 15/15
Val Accuracy: 0.5000, Val F1: 0.3333

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.5080
F1 Score: 0.4547


In [25]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-4,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'ratio_100', 'ratio_100', CLASS_B),
        "val_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_A),
        "val_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/val_data',
                              CLASS_B),
        "test_A": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_A),
        "test_B": os.path.join('/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/datasets/pneumonia_cxr/test_data',
                               CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_resnet18.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.2500, Val F1: 0.2667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5625, Val F1: 0.6018
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.6250, Val F1: 0.6250
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.6250, Val F1: 0.6250

🚀 Epoch 5/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 6/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 7/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 8/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 9/15
Val Accuracy: 0.6250, Val F1: 0.6250

🚀 Epoch 10/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 11/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 12/15
Val Accuracy: 0.6250, Val F1: 0.6250

🚀 Epoch 13/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 14/15
Val Accuracy: 0.5625, Val F1: 0.5608

🚀 Epoch 15/15
Val Accuracy: 0.5625, Val F1: 0.5608

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.6058
F1 Score: 0.5895
